# Grouped hyperparameter optimisation

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load development data

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Randomised search with patient-grouped folds

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from src.models.boosting import models
m=make_model(X,models()['XGBoost'],quick=QUICK)
search=RandomizedSearchCV(m,{'model__max_depth':[2,3,4,5],'model__n_estimators':[20,40,80,150],'model__learning_rate':[.03,.08,.15],'model__reg_lambda':[1,3,8]},n_iter=2 if QUICK else 10,scoring='average_precision',cv=StratifiedGroupKFold(3,shuffle=True,random_state=45),random_state=42,n_jobs=1)
search.fit(X,Y,groups=G); print(search.best_params_,search.best_score_)
r=pd.DataFrame(search.cv_results_); display(r[['params','mean_test_score','std_test_score','rank_test_score']]); r.to_csv(OUT/'random_search.csv',index=False)

Compare validation performance

In [ ]:
base=make_model(X,models()['XGBoost'],quick=QUICK).fit(X,Y)
for name,m in [('default',base),('tuned',search.best_estimator_)]: print(name,scores(y.iloc[tune],m.predict_proba(x.iloc[tune])[:,1]))
r.mean_test_score.plot.bar(title='CV average precision'); plt.show()